# 04 - Evaluate generated molecules against the paper's tables

Reads `outputs/samples/<fold>/run*/protein_*.csv` from `03_generate_samples.py` and
reproduces, as far as this pipeline allows:

| Paper | Here |
|---|---|
| **Table 1** drug-like properties | fully reproducible |
| **Table 2** target awareness (docking) | `05_docking_table2.py` - needs SMINA + receptors |
| **Table 3** % toxic by **pkCSM** | **substituted** with the repo's own Ames RF, plus a pkCSM upload file |
| CDGCN Table 1 novelty, generation time | computed here as extras |

**Aggregation follows the paper**: a percentage is computed per protein, averaged over
the held-out proteins to give one number per run, then reported as mean +/- std over
the 10 runs. That is what makes the published error bars as small as +/-0.3.

**Denominator**: numerator = unique valid molecules satisfying the property,
denominator = molecules *requested* (100). This is the only reading consistent with
the paper's own numbers - in Table 1 validity equals uniqueness for CDGCN and CTDDG,
and every SSVAE property row equals its validity x uniqueness (0.884 x 58.1 = 51.4).

**Table 3 is not reproducible as published.** It is pkCSM; this repo labels with its
own Ames RF. Worse, that RF produced the training labels, so "% toxic by the RF" is
partly scoring the model against its own supervision. Read the RF column as an
internal diagnostic and use `pkcsm_upload.smi` for the real comparison.

In [1]:
import os
import re
import csv
import glob
import json
import time
import pickle
import hashlib
import warnings
from collections import defaultdict

import numpy as np
import pandas as pd

import rdkit
from rdkit import Chem, RDConfig, RDLogger
from rdkit.Chem import Descriptors, Crippen, Lipinski, QED

RDLogger.DisableLog('rdApp.*')
warnings.filterwarnings('ignore')

import sys
sys.path.append(os.path.join(RDConfig.RDContribDir, 'SA_Score'))
import sascorer  # noqa: E402

print(f"rdkit {rdkit.__version__}")

rdkit 2026.03.6


## Config

In [ ]:
REPO_ROOT = os.environ.get("CTDDG_REPO_ROOT", "/home/muhamed/repo/CTDDG")
FOLD = os.environ.get("CTDDG_FOLD", "d1")

SAMPLES_DIR = os.path.join(REPO_ROOT, "outputs/samples", FOLD)
BINDINGDB_DIR = os.path.join(REPO_ROOT, "datasets/data/bindingdb")
RF_MODEL_PATH = os.path.join(REPO_ROOT, "models/ames_random_forest.pkl")
FP_CONFIG_PATH = os.path.join(REPO_ROOT, "models/fingerprint_config.pkl")
TRAIN_AMES_META = os.path.join(BINDINGDB_DIR, "ligand_ames_labels.csv.meta.json")

EVAL_DIR = os.path.join(REPO_ROOT, "outputs/evaluation", FOLD)
os.makedirs(EVAL_DIR, exist_ok=True)

GEN_AMES_CACHE = os.path.join(EVAL_DIR, "generated_ames_labels.csv")
NOVELTY_CACHE = os.path.join(EVAL_DIR, f"{FOLD}_dataset_canonical_smiles.txt")

# Optional {"4nos": <protein_row>, ...} so per-PDB rows of Tables 2/3 can be filled in.
# 05_docking_table2.py writes this; absent is fine.
PDB_MAP_PATH = os.path.join(EVAL_DIR, "pdb_protein_map.json")

COMPUTE_TOXICITY = True
COMPUTE_NOVELTY = True       # canonicalises ~144k dataset ligands once, then caches

FOLD_SUFFIX = {"d1": "4_org_1042_104", "d2": "4_org_1000_112", "d3": "4_org_1004_122",
               "d4": "4_org_1002_103", "d5": "4_org_1036_124"}

# Paper thresholds (CTDDG Table 1 / Section 3.2.1).
THRESHOLDS = {
    'logp_lt_5':      ('LogP < 5',                 lambda d: d['logp'] < 5),
    'mw_lt_500':      ('Molecular weight < 500',   lambda d: d['mw'] < 500),
    'hbd_lt_5':       ('H-bond donors < 5',        lambda d: d['hbd'] < 5),
    'hba_lt_10':      ('H-bond acceptors < 10',    lambda d: d['hba'] < 10),
    'rotb_lt_10':     ('Rotatable bonds < 10',     lambda d: d['rotb'] < 10),
    'tpsa_lt_140':    ('TPSA < 140',               lambda d: d['tpsa'] < 140),
    'sas_lt_6':       ('SAS < 6',                  lambda d: d['sas'] < 6),
}

# CTDDG Table 1, for side-by-side output. Not used in any computation.
PAPER_TABLE1 = {
    'validity':    {'CDGCN': (91.7, 0.22), 'SSVAE': (88.4, 3.29), 'CTDDG': (94.4, 0.32)},
    'uniqueness':  {'CDGCN': (91.7, 0.22), 'SSVAE': (58.1, 4.13), 'CTDDG': (94.4, 0.32)},
    'logp_lt_5':   {'CDGCN': (68.6, 0.38), 'SSVAE': (51.4, 4.31), 'CTDDG': (74.8, 0.33)},
    'mw_lt_500':   {'CDGCN': (73.5, 0.47), 'SSVAE': (51.4, 4.31), 'CTDDG': (71.1, 0.32)},
    'hbd_lt_5':    {'CDGCN': (84.6, 0.25), 'SSVAE': (51.4, 4.31), 'CTDDG': (87.9, 0.40)},
    'hba_lt_10':   {'CDGCN': (84.5, 0.28), 'SSVAE': (51.4, 4.31), 'CTDDG': (87.2, 0.43)},
    'rotb_lt_10':  {'CDGCN': (80.3, 0.40), 'SSVAE': (51.4, 4.31), 'CTDDG': (79.9, 0.38)},
    'tpsa_lt_140': {'CDGCN': (83.5, 0.23), 'SSVAE': (51.4, 4.31), 'CTDDG': (84.1, 0.36)},
    'qed':         {'CDGCN': (0.55, 0.00), 'SSVAE': (0.44, 0.02), 'CTDDG': (0.51, 0.00)},
    'sas_lt_6':    {'CDGCN': (84.7, 0.23), 'SSVAE': (51.4, 4.31), 'CTDDG': (88.6, 0.37)},
}

# CTDDG Table 3, % toxic by pkCSM.
PAPER_TABLE3 = {'4nos': 19.84, '2nru': 23.32, '1lqf': 20.26, '1k2r': 20.02,
                '1cqp': 13.17, 'Avg': 19.32}
PAPER_TABLE3_CDGCN = {'4nos': 24.13, '2nru': 26.89, '1lqf': 23.12, '1k2r': 22.49,
                      '1cqp': 19.88, 'Avg': 23.30}

ROW_ORDER = ['validity', 'uniqueness', 'logp_lt_5', 'mw_lt_500', 'hbd_lt_5', 'hba_lt_10',
             'rotb_lt_10', 'tpsa_lt_140', 'qed', 'sas_lt_6']
ROW_LABEL = {
    'validity': 'Validity (%)', 'uniqueness': 'Uniqueness (%)',
    'logp_lt_5': 'LogP < 5 (%)', 'mw_lt_500': 'Molecular weight < 500 (%)',
    'hbd_lt_5': 'H-bond donors < 5 (%)', 'hba_lt_10': 'H-bond acceptors < 10 (%)',
    'rotb_lt_10': 'Rotatable bonds < 10 (%)', 'tpsa_lt_140': 'TPSA < 140 (%)',
    'qed': 'QED (absolute)', 'sas_lt_6': 'SAS < 6 (%)',
}

## Load generated samples

In [3]:
manifest_path = os.path.join(SAMPLES_DIR, "manifest.json")
if not os.path.isfile(manifest_path):
    raise FileNotFoundError(f"{manifest_path} not found - run 03_generate_samples.py first.")
with open(manifest_path) as f:
    MANIFEST = json.load(f)

N_SAMPLES = int(MANIFEST['n_samples_per_protein'])
N_RUNS = int(MANIFEST['n_runs'])

print(json.dumps({k: MANIFEST[k] for k in
                  ('fold', 'n_proteins_sampled', 'n_held_out_proteins_total',
                   'n_samples_per_protein', 'n_runs', 'generated_at')}, indent=1))

if MANIFEST.get('bundle_smoke_test') or MANIFEST.get('checkpoint_meta', {}).get('smoke_test'):
    print("\n[WARN] smoke artifacts in the lineage - these metrics are not reportable.")
if MANIFEST['n_proteins_sampled'] != MANIFEST['n_held_out_proteins_total']:
    print(f"\n[WARN] {MANIFEST['n_proteins_sampled']} of "
          f"{MANIFEST['n_held_out_proteins_total']} held-out proteins sampled. The paper "
          f"averages over all of them.")


def load_samples():
    """-> {(run, protein_row): [smiles or '' ...]} with one entry per requested sample."""
    out = {}
    for run_dir in sorted(glob.glob(os.path.join(SAMPLES_DIR, "run*"))):
        run = int(re.search(r"run(\d+)$", run_dir).group(1))
        for path in sorted(glob.glob(os.path.join(run_dir, "protein_*.csv"))):
            row = int(re.search(r"protein_(\d+)\.csv$", path).group(1))
            with open(path) as f:
                smis = [r['smiles'] for r in csv.DictReader(f)]
            if len(smis) != N_SAMPLES:
                print(f"[WARN] {os.path.basename(path)} (run {run}) has {len(smis)} rows, "
                      f"expected {N_SAMPLES}")
            out[(run, row)] = smis
    return out


SAMPLES = load_samples()
runs_present = sorted({r for r, _ in SAMPLES})
proteins_present = sorted({p for _, p in SAMPLES})
print(f"\nloaded {len(SAMPLES)} (run, protein) cells | runs {runs_present} | "
      f"{len(proteins_present)} proteins")
if len(runs_present) < N_RUNS:
    print(f"[WARN] only {len(runs_present)} of {N_RUNS} runs present; error bars will be "
          f"wider than the paper's.")

{
 "fold": "d1",
 "n_proteins_sampled": 104,
 "n_held_out_proteins_total": 104,
 "n_samples_per_protein": 100,
 "n_runs": 10,
 "generated_at": "2026-10-11 19:47:05"
}

loaded 1040 (run, protein) cells | runs [1, 2, 3, 4, 5, 6, 7, 8, 9, 10] | 104 proteins


## Descriptors

In [4]:
_DESC_CACHE = {}


def descriptors(smiles):
    """Cached per canonical SMILES. None if RDKit cannot sanitize it."""
    if smiles in _DESC_CACHE:
        return _DESC_CACHE[smiles]
    mol = Chem.MolFromSmiles(smiles) if smiles else None
    if mol is None:
        _DESC_CACHE[smiles] = None
        return None
    try:
        d = {
            'logp': Crippen.MolLogP(mol),
            'mw': Descriptors.MolWt(mol),
            'hbd': Lipinski.NumHDonors(mol),
            'hba': Lipinski.NumHAcceptors(mol),
            'rotb': Lipinski.NumRotatableBonds(mol),
            'tpsa': Descriptors.TPSA(mol),
            'qed': QED.qed(mol),
            'sas': sascorer.calculateScore(mol),
            'n_atoms': mol.GetNumAtoms(),
        }
    except Exception:
        d = None
    _DESC_CACHE[smiles] = d
    return d


def cell_metrics(smiles_list, n_requested=None):
    """Paper-protocol metrics for one (protein, run): 100 requested molecules.

    valid       = RDKit-sanitizable
    unique      = distinct canonical SMILES among the valid ones
    property %  = unique-valid satisfying it, over n_requested
    """
    n_requested = n_requested or len(smiles_list)
    valid = [s for s in smiles_list if s and descriptors(s) is not None]
    unique = list(dict.fromkeys(valid))

    m = {'n_requested': n_requested,
         'n_valid': len(valid),
         'n_unique': len(unique),
         'validity': 100.0 * len(valid) / n_requested,
         'uniqueness': 100.0 * len(unique) / n_requested,
         # secondary convention, reported alongside but not used for the paper table
         'uniqueness_among_valid': 100.0 * len(unique) / max(1, len(valid))}

    ds = [descriptors(s) for s in unique]
    for key, (_, test) in THRESHOLDS.items():
        m[key] = 100.0 * sum(1 for d in ds if test(d)) / n_requested
    m['qed'] = float(np.mean([d['qed'] for d in ds])) if ds else float('nan')
    m['mean_n_atoms'] = float(np.mean([d['n_atoms'] for d in ds])) if ds else float('nan')
    return m


t0 = time.time()
cells = []
for (run, row), smis in sorted(SAMPLES.items()):
    rec = {'run': run, 'protein_row': row}
    rec.update(cell_metrics(smis, N_SAMPLES))
    cells.append(rec)
CELLS = pd.DataFrame(cells)
print(f"descriptors for {len(_DESC_CACHE)} distinct SMILES in {time.time() - t0:.1f}s")
CELLS.to_csv(os.path.join(EVAL_DIR, "per_protein_per_run.csv"), index=False)

descriptors for 24787 distinct SMILES in 36.7s


## Table 1

Average over proteins within a run, then mean +/- std across runs.

In [5]:
METRIC_COLS = ROW_ORDER + ['uniqueness_among_valid', 'mean_n_atoms']
per_run = CELLS.groupby('run')[METRIC_COLS].mean()


def mean_std(col):
    v = per_run[col].to_numpy(dtype=float)
    return float(np.mean(v)), float(np.std(v, ddof=1)) if len(v) > 1 else 0.0


table1 = []
for key in ROW_ORDER:
    mu, sd = mean_std(key)
    paper = PAPER_TABLE1[key]
    table1.append({
        'Property': ROW_LABEL[key],
        'This run': f"{mu:.2f} +/- {sd:.2f}" if key != 'qed' else f"{mu:.3f} +/- {sd:.3f}",
        'CTDDG (paper)': f"{paper['CTDDG'][0]} +/- {paper['CTDDG'][1]}",
        'CDGCN (paper)': f"{paper['CDGCN'][0]} +/- {paper['CDGCN'][1]}",
        'SSVAE (paper)': f"{paper['SSVAE'][0]} +/- {paper['SSVAE'][1]}",
        '_value': mu, '_std': sd,
    })
TABLE1 = pd.DataFrame(table1)

print("\n=== Table 1: drug-like properties ===")
print(f"{len(proteins_present)} proteins x {len(runs_present)} runs x {N_SAMPLES} samples")
print(TABLE1.drop(columns=['_value', '_std']).to_string(index=False))

mu_uv, sd_uv = mean_std('uniqueness_among_valid')
mu_at, sd_at = mean_std('mean_n_atoms')
print(f"\nsecondary: uniqueness among valid {mu_uv:.2f} +/- {sd_uv:.2f} % | "
      f"mean heavy atoms {mu_at:.1f} +/- {sd_at:.1f}")

TABLE1.drop(columns=['_value', '_std']).to_csv(
    os.path.join(EVAL_DIR, "table1_drug_like_properties.csv"), index=False)


=== Table 1: drug-like properties ===
104 proteins x 10 runs x 100 samples
                  Property        This run CTDDG (paper) CDGCN (paper) SSVAE (paper)
              Validity (%)  23.88 +/- 0.40 94.4 +/- 0.32 91.7 +/- 0.22 88.4 +/- 3.29
            Uniqueness (%)  23.87 +/- 0.41 94.4 +/- 0.32 91.7 +/- 0.22 58.1 +/- 4.13
              LogP < 5 (%)  17.73 +/- 0.39 74.8 +/- 0.33 68.6 +/- 0.38 51.4 +/- 4.31
Molecular weight < 500 (%)  17.76 +/- 0.36 71.1 +/- 0.32 73.5 +/- 0.47 51.4 +/- 4.31
     H-bond donors < 5 (%)  22.66 +/- 0.40  87.9 +/- 0.4 84.6 +/- 0.25 51.4 +/- 4.31
 H-bond acceptors < 10 (%)  23.31 +/- 0.41 87.2 +/- 0.43 84.5 +/- 0.28 51.4 +/- 4.31
  Rotatable bonds < 10 (%)  17.07 +/- 0.37 79.9 +/- 0.38  80.3 +/- 0.4 51.4 +/- 4.31
            TPSA < 140 (%)  21.93 +/- 0.41 84.1 +/- 0.36 83.5 +/- 0.23 51.4 +/- 4.31
            QED (absolute) 0.495 +/- 0.007  0.51 +/- 0.0  0.55 +/- 0.0 0.44 +/- 0.02
               SAS < 6 (%)  23.33 +/- 0.44 88.6 +/- 0.37 84.7 +/- 0.23 51.

## Toxicity (Table 3 substitute)

The repo's Ames RF, not pkCSM. The RF hash is checked against the one that wrote the
training labels: a mismatch means the generated molecules and the training ligands were
scored by two different definitions of toxic.

In [6]:
def _load_pickle(path):
    errors = []
    for how, fn in (("pickle", lambda: pickle.load(open(path, 'rb'))),
                    ("joblib", lambda: __import__('joblib').load(path))):
        try:
            return fn()
        except Exception as e:
            errors.append(f"{how}: {type(e).__name__}: {e}")
    raise RuntimeError(f"Could not load {path}\n  " + "\n  ".join(errors))


def _sha256(path, chunk=1 << 20):
    h = hashlib.sha256()
    with open(path, 'rb') as f:
        for block in iter(lambda: f.read(chunk), b''):
            h.update(block)
    return h.hexdigest()


def morgan_matrix(smiles_list, radius, n_bits):
    try:
        from rdkit.Chem import rdFingerprintGenerator
        gen = rdFingerprintGenerator.GetMorganGenerator(radius=radius, fpSize=n_bits)
        fp_of = gen.GetFingerprint
    except (ImportError, AttributeError):
        from rdkit.Chem import AllChem
        fp_of = lambda m: AllChem.GetMorganFingerprintAsBitVect(m, radius, nBits=n_bits)

    out = np.zeros((len(smiles_list), n_bits), dtype=np.uint8)
    for i, smi in enumerate(smiles_list):
        mol = Chem.MolFromSmiles(smi)
        if mol is not None:
            out[i] = np.frombuffer(fp_of(mol).ToBitString().encode(), dtype=np.uint8) - ord('0')
    return out


def ames_labels(smiles_list):
    """{smiles: 0/1} from the repo's Ames RF, cached in EVAL_DIR."""
    cache = {}
    if os.path.isfile(GEN_AMES_CACHE):
        df = pd.read_csv(GEN_AMES_CACHE)
        cache = dict(zip(df['smiles'].astype(str), df['ames'].astype(int)))
        print(f"[INFO] Ames cache: {len(cache)} entries")

    todo = sorted({s for s in smiles_list if s and s not in cache})
    if todo:
        cfg = _load_pickle(FP_CONFIG_PATH)
        radius = int(cfg.get('radius', 2))
        n_bits = int(cfg.get('n_bits', cfg.get('nBits', 2048)))
        rf = _load_pickle(RF_MODEL_PATH)
        print(f"[INFO] labelling {len(todo)} molecules | Morgan r{radius}, {n_bits} bits")

        X = morgan_matrix(todo, radius, n_bits)
        if getattr(rf, 'n_features_in_', n_bits) != n_bits:
            raise ValueError(f"RF expects {rf.n_features_in_} features, config gives {n_bits}")
        cache.update(zip(todo, rf.predict(X).astype(int)))
        pd.DataFrame({'smiles': list(cache), 'ames': list(cache.values())}) \
          .to_csv(GEN_AMES_CACHE, index=False)
    return cache


TOX = {}
if COMPUTE_TOXICITY and os.path.isfile(RF_MODEL_PATH):
    # Same-forest check against the labels Eq. 8 was trained on.
    if os.path.isfile(TRAIN_AMES_META):
        with open(TRAIN_AMES_META) as f:
            train_fp = json.load(f)
        now_sha = _sha256(RF_MODEL_PATH)
        train_sha = (train_fp.get('rf') or {}).get('sha256')
        if train_sha and train_sha != now_sha:
            print("[WARN] the RF on disk is not the one that labelled the training ligands. "
                  "Generated and training molecules would be scored by different forests.")
        elif train_sha:
            print("[OK] same Ames RF as the training labels.")

    all_smiles = [s for smis in SAMPLES.values() for s in smis if s]
    TOX = ames_labels(all_smiles)
elif COMPUTE_TOXICITY:
    print(f"[WARN] no RF at {RF_MODEL_PATH}; skipping the toxicity table. Regenerate it "
          f"from code/random_forest_classifier.ipynb (random_state=42).")

[OK] same Ames RF as the training labels.
[INFO] Ames cache: 24787 entries


In [7]:
if TOX:
    rows = []
    for (run, row), smis in sorted(SAMPLES.items()):
        unique_valid = list(dict.fromkeys(
            [s for s in smis if s and descriptors(s) is not None]))
        labelled = [TOX[s] for s in unique_valid if s in TOX]
        rows.append({'run': run, 'protein_row': row,
                     'n_scored': len(labelled),
                     'pct_toxic': 100.0 * np.mean(labelled) if labelled else float('nan')})
    TOXDF = pd.DataFrame(rows)
    TOXDF.to_csv(os.path.join(EVAL_DIR, "toxicity_per_protein_per_run.csv"), index=False)

    per_run_tox = TOXDF.groupby('run')['pct_toxic'].mean().to_numpy(dtype=float)
    tox_mu = float(np.mean(per_run_tox))
    tox_sd = float(np.std(per_run_tox, ddof=1)) if len(per_run_tox) > 1 else 0.0

    print("\n=== Table 3 substitute: % predicted toxic ===")
    print("Labeller here: the repo's Ames Random Forest. The paper's Table 3 is pkCSM, so "
          "the absolute numbers below are NOT comparable to it.")
    print(f"\nAll {len(proteins_present)} held-out proteins: {tox_mu:.2f} +/- {tox_sd:.2f} %")
    print(f"  paper, 5 PDBs, pkCSM:  CTDDG {PAPER_TABLE3['Avg']} | "
          f"CDGCN {PAPER_TABLE3_CDGCN['Avg']}")

    # Per-PDB rows, if the mapping from 05_docking_table2.py exists.
    if os.path.isfile(PDB_MAP_PATH):
        with open(PDB_MAP_PATH) as f:
            pdb_map = json.load(f)
        sub = []
        for pdb, prow in pdb_map.items():
            v = TOXDF[TOXDF['protein_row'] == int(prow)].groupby('run')['pct_toxic'].mean()
            if len(v) == 0:
                continue
            sub.append({'PDB': pdb,
                        'This run (RF)': f"{v.mean():.2f} +/- "
                                         f"{v.std(ddof=1) if len(v) > 1 else 0.0:.2f}",
                        'CTDDG paper (pkCSM)': PAPER_TABLE3.get(pdb, '-'),
                        'CDGCN paper (pkCSM)': PAPER_TABLE3_CDGCN.get(pdb, '-')})
        if sub:
            print()
            print(pd.DataFrame(sub).to_string(index=False))
            pd.DataFrame(sub).to_csv(
                os.path.join(EVAL_DIR, "table3_toxicity.csv"), index=False)
    else:
        print(f"\n[INFO] no {os.path.basename(PDB_MAP_PATH)}; per-PDB rows skipped. "
              f"Run 05_docking_table2.py to write it.")

    # pkCSM takes a plain SMILES list. Dedup hard - the server is rate-limited.
    pk_path = os.path.join(EVAL_DIR, "pkcsm_upload.smi")
    uniq_all = sorted({s for smis in SAMPLES.values() for s in smis
                       if s and descriptors(s) is not None})
    with open(pk_path, 'w') as f:
        f.write("\n".join(uniq_all) + "\n")
    print(f"\n[INFO] {len(uniq_all)} unique valid SMILES -> {pk_path} "
          f"(for pkCSM, if you want Table 3's actual metric)")


=== Table 3 substitute: % predicted toxic ===
Labeller here: the repo's Ames Random Forest. The paper's Table 3 is pkCSM, so the absolute numbers below are NOT comparable to it.

All 104 held-out proteins: 6.33 +/- 0.33 %
  paper, 5 PDBs, pkCSM:  CTDDG 19.32 | CDGCN 23.3

[INFO] no pdb_protein_map.json; per-PDB rows skipped. Run 05_docking_table2.py to write it.

[INFO] 24787 unique valid SMILES -> /home/muhamed/repo/CTDDG/outputs/evaluation/d1/pkcsm_upload.smi (for pkCSM, if you want Table 3's actual metric)


## Novelty

CDGCN Table 1's definition: exact match against the dataset, on canonical SMILES.

In [8]:
def dataset_canonical_smiles():
    if os.path.isfile(NOVELTY_CACHE):
        with open(NOVELTY_CACHE) as f:
            return {ln.strip() for ln in f if ln.strip()}

    suffix = FOLD_SUFFIX[FOLD]
    paths = []
    for split in ("train", "test"):
        d = os.path.join(BINDINGDB_DIR, f"{split}_dataset", f"{split}_{suffix}")
        paths += sorted(glob.glob(os.path.join(d, "ligands_*_corrected")))
    if not paths:
        print(f"[WARN] no ligand pair files for fold {FOLD}; skipping novelty.")
        return None

    raw = set()
    for p in paths:
        with open(p) as f:
            for line in f:
                s = ''.join(line.split())
                if s:
                    raw.add(s)
    print(f"[INFO] canonicalising {len(raw)} dataset ligands (once, then cached)")

    canon = set()
    for s in raw:
        m = Chem.MolFromSmiles(s)
        if m is not None:
            canon.add(Chem.MolToSmiles(m))
    with open(NOVELTY_CACHE, 'w') as f:
        f.write("\n".join(sorted(canon)) + "\n")
    return canon


if COMPUTE_NOVELTY:
    DATASET_SMILES = dataset_canonical_smiles()
    if DATASET_SMILES:
        rows = []
        for (run, row), smis in sorted(SAMPLES.items()):
            uv = list(dict.fromkeys([s for s in smis if s and descriptors(s) is not None]))
            novel = [s for s in uv if s not in DATASET_SMILES]
            rows.append({'run': run, 'protein_row': row,
                         'novelty': 100.0 * len(novel) / max(1, len(uv))})
        NOV = pd.DataFrame(rows).groupby('run')['novelty'].mean().to_numpy(dtype=float)
        print(f"\nNovelty (unique valid not in the fold's ligand set): "
              f"{np.mean(NOV):.2f} +/- {np.std(NOV, ddof=1) if len(NOV) > 1 else 0.0:.2f} %")
        print(f"  CDGCN paper, novel proteins, S_1000: 99.8 +/- 0.05 %")
        print(f"  Note: stage 1 trained on the UNFILTERED ChEMBL export, which still "
              f"contains these ligands. High novelty here does not clear that leak.")


Novelty (unique valid not in the fold's ligand set): 100.00 +/- 0.00 %
  CDGCN paper, novel proteins, S_1000: 99.8 +/- 0.05 %
  Note: stage 1 trained on the UNFILTERED ChEMBL export, which still contains these ligands. High novelty here does not clear that leak.


## Generation time

CDGCN Table 1 reports time to produce S_10 / S_100 / S_1000. Only measured if
`timing.csv` is present.

In [9]:
timing_path = os.path.join(SAMPLES_DIR, "timing.csv")
if os.path.isfile(timing_path):
    tdf = pd.read_csv(timing_path)
    per_mol = tdf['seconds'] / tdf['n_samples']
    print(f"\n{per_mol.mean() * 1000:.0f} ms per molecule "
          f"({MANIFEST['device']}), so S_10 ~ {per_mol.mean() * 10:.1f}s, "
          f"S_100 ~ {per_mol.mean() * 100:.1f}s, S_1000 ~ {per_mol.mean() * 1000:.0f}s")


9 ms per molecule (cuda:0), so S_10 ~ 0.1s, S_100 ~ 0.9s, S_1000 ~ 9s


## Report

In [10]:
report = [
    f"# CTDDG reproduction - evaluation, fold {FOLD}",
    "",
    f"Generated {MANIFEST['generated_at']} | checkpoint `{MANIFEST['checkpoint']}`",
    f"{len(proteins_present)} held-out proteins x {len(runs_present)} runs x "
    f"{N_SAMPLES} samples = {len(proteins_present) * len(runs_present) * N_SAMPLES} molecules",
    "",
    "## Why these numbers are not a reproduction of Tables 1-3",
    "",
    "- Stage 1 trained on the unfiltered ChEMBL export (1,207,359 molecules), which still "
    "contains the BindingDB ligands. The fine-tuning validation and held-out test ligands "
    "leaked into pretraining; both papers remove them.",
    "- Stage 1 ran at `BATCH_SIZE = 16`; the paper used 8 in both phases.",
    "- Fine-tuning did not converge - best epoch was 1 and non-toxic validation NLL "
    "degraded after it.",
    "- BindingDB ligands were labelled by the repo's Ames RF, not pkCSM, so Table 3's "
    "metric is substituted rather than reproduced.",
    "- No CDGCN column: a conditional CDGCN fine-tune on this fold does not exist here, so "
    "the paper's central comparison cannot be made yet.",
    "",
    "## Table 1 - drug-like properties",
    "",
    TABLE1.drop(columns=['_value', '_std']).to_markdown(index=False),
    "",
    f"Secondary: uniqueness among valid molecules {mu_uv:.2f} +/- {sd_uv:.2f} %; "
    f"mean heavy atoms {mu_at:.1f}.",
    "",
    "Numerator is unique valid molecules satisfying the property; denominator is the "
    f"{N_SAMPLES} requested. Percentages are averaged over proteins within a run, then "
    "reported as mean +/- std over runs.",
    "",
]
if TOX:
    report += [
        "## Table 3 substitute - predicted toxicity",
        "",
        f"Repo Ames RF, all held-out proteins: **{tox_mu:.2f} +/- {tox_sd:.2f} %** toxic.",
        "",
        f"Paper (pkCSM, 5 PDBs): CTDDG {PAPER_TABLE3['Avg']}, "
        f"CDGCN {PAPER_TABLE3_CDGCN['Avg']}. Different labeller, different protein subset - "
        "do not read the gap as a result. The RF also produced the training labels, so this "
        "column partly scores the model against its own supervision.",
        "",
        f"`pkcsm_upload.smi` holds the unique valid SMILES for the real metric.",
        "",
    ]
report += [
    "## Table 2 - target awareness",
    "",
    "Needs docking. See `05_docking_table2.py`.",
    "",
]

report_path = os.path.join(EVAL_DIR, "report.md")
with open(report_path, 'w') as f:
    f.write("\n".join(report))

print(f"\nreport  {report_path}")
print(f"tables  {EVAL_DIR}")


report  /home/muhamed/repo/CTDDG/outputs/evaluation/d1/report.md
tables  /home/muhamed/repo/CTDDG/outputs/evaluation/d1
